# Engineered token structures and their analytical limits

Notebook 02 measured token-string length only. Here we test the observed angle-bracket field grammar and tag presence, then adjust anomaly comparisons for payment composition. Literal tokens, accounts, and payment details are never exported. Implementation: `TOKEN_RE`, `token_features`, and `tokens_analysis` in `src/remaining_analysis.py`.

The parser reads `<field:value>` blocks and permits colons within values; separators outside blocks are preserved in the source. The strict regex is a tested serialization hypothesis. A first execution rejected universal balanced blocks: 1,650 opening brackets were unmatched, all in chr fields. Field-opening markers are therefore parsed independently and validated against every opening bracket; strict-block residuals remain evidence. Parsed values are used only for a local case-insensitive literal substring diagnostic against source details. No identity, entity resolution, or semantic accuracy is inferred.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "data/cleaned/swift_payments_cleaned.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / ".venv/Scripts/python.exe").exists()
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve(), "Use the project virtual environment"
sys.path.insert(0, str(ROOT / "src"))
from remaining_analysis import Study, tokens_analysis
study = Study(ROOT, '07_tokens', prerequisites=('06_anomaly_analysis',))

## Serialization coverage and structural vocabulary

**Purpose.** Determine which token patterns can be measured reproducibly.

**Hypothesis.** Every field opening has a recognizable field name; universal balanced blocks are a separate hypothesis tested and revised from the first failed run. Field-count distributions vary by party pair.

**Methodology.** One chunked scan reads both token and detail fields. Validate every field-opening marker and report strict-block residuals on both bracket types. Export structural field names and per-transaction presence counts only; repeated field names in one string count once in presence tables. Report blank strings, zero parsed fields, character counts, and literal-value hit counts. Denominator for overall presence is all 1,048,575 rows per side.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** Complete field-opening coverage supports field-count measurement; nonzero strict-block residuals reject universal balanced serialization. Literal mismatch is compatible with shape encoding, transliteration or generator conventions and is not an anomaly verdict.

**Limitations.** Bracket validation does not prove all bytes or values are semantically meaningful. Unknown field semantics require organizer documentation. Literal matches can be accidental, especially short values.

**Business Implications.** Structural token QA can identify formatting inconsistencies without exposing personal-shaped synthetic strings.

In [ ]:
quality, tag_counts, profiles, effects, associations = tokens_analysis(study)
display(quality)
display(tag_counts)
display(associations)
display(profiles[0])
display(Image(filename=str(study.charts / "token_party_structure.png")))

## Anomaly associations and confounding by party composition

**Purpose.** Test whether label differences survive payment-segment adjustment.

**Hypothesis.** Ordering or beneficiary token field counts differ between each observed anomaly label and UNLABELED after party, route, currency and amount-band adjustment.

**Methodology.** Show all label profiles and all predeclared label-side contrasts. Exact strata require at least ten rows in each arm. Overlap-weighted mean field-count differences and analytic 95% intervals are in parsed fields. Full tag-by-party and tag-by-label presence tables show the denominator for each group; absent combinations have zero presence, not unknown values.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** Party-dependent structures may explain apparent anomaly effects. Negative and positive strata are both reported; a direction reversal limits any universal claim.

**Limitations.** The contrasts are exploratory and numerous; intervals are pointwise, not familywise. No token value leakage is used for a fraud model. Shared synthetic templates can violate independent-draw interval assumptions.

**Business Implications.** Focus follow-up on stable structural branches; do not deploy token formatting as a fraud signal without external validation.

In [ ]:
display(profiles[1])
display(effects)
display(pd.read_csv(study.tables / "token_tag_presence_by_party.csv").head(30))

In [ ]:
report = study.finish([f"{r.side}: {r.n:,} strings, {r.blank_n:,} blank, {r.zero_parsed_tags_n:,} without parsed fields; mean field count {r.mean_tags:.6f}; unmatched opening/closing brackets {r.unparsed_opening_brackets}/{r.unparsed_closing_brackets}." for r in quality.itertuples()] +
                      [f"{r.right} versus party: Cramér V {r.cramers_v:.6f}." for r in associations.itertuples()] +
                      ["Identity inconsistency from literal token/detail mismatch is rejected as an unsupported interpretation; shape encoding remains an alternative."], ['Field semantics are inferred from syntax only.', 'Pointwise label contrasts face multiple-comparison risk and residual confounding.', 'Literal values and account identifiers remain private; parsing does not establish identity or fraud.'], 'Validate structural and statistical rules using independent random and chronological splits; reconcile earlier candidates.')
display(Markdown("### Final findings and limitations\n" + "\n".join("- " + x for x in report["findings"]) + "\n\n" + " ".join(report["limitations"])))
display(pd.DataFrame(study.checks))
display(Markdown("### Completion summary\nExecution passed for " + format(report["population"], ",") + " transactions. All generated files are listed below. No unresolved material errors.\n\n" + "\n".join("- `"+x+"`" for x in report["files"]) + "\n\nRecommended next investigation: " + report["next_investigation"]))